# Direct Gold Cross Crypto Macro Features 1d

Thin notebook entrypoint for the phase-1 cross-domain Gold feature product.

This notebook delegates market-feature assembly, macro observation-date as-of alignment, MAP feature construction, Delta merge logic, and observability writes to `src/lakehouse`.

Fill contract:

- `fill_policy = asof_observation_date_ffill_v1`
- suitable for analytics and dashboarding
- not declared backtest-safe for macro release timing or revision availability


In [ ]:
import sys
import uuid
from pathlib import Path


def add_repo_src_to_path() -> str:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        src_dir = candidate / "src"
        if (src_dir / "lakehouse" / "__init__.py").exists():
            src_dir_str = str(src_dir)
            if src_dir_str not in sys.path:
                sys.path.insert(0, src_dir_str)
            return src_dir_str

    raise RuntimeError("Could not locate the repository src directory from the current notebook path")


REPO_SRC = add_repo_src_to_path()

from lakehouse.pipelines.gold import run_gold_cross_crypto_macro_features

spark.conf.set("spark.sql.session.timeZone", "UTC")

dbutils.widgets.text("catalog", "market_macro")
dbutils.widgets.text("product_ids", "BTC-USD,ETH-USD,SOL-USD,LINK-USD,DOGE-USD")
dbutils.widgets.text("macro_indicator_ids", "ECB_FX_REF_EUR_USD,FRED_CPIAUCSL,FRED_FEDFUNDS,FRED_GDP")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "backfill"])
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")
dbutils.widgets.text("lookback_days", "120")
dbutils.widgets.text("run_id", str(uuid.uuid4()))


In [ ]:
result = run_gold_cross_crypto_macro_features(
    spark=spark,
    raw_product_ids=dbutils.widgets.get("product_ids").strip(),
    raw_macro_indicator_ids=dbutils.widgets.get("macro_indicator_ids").strip(),
    mode=dbutils.widgets.get("mode").strip().lower(),
    start_date_raw=dbutils.widgets.get("start_date").strip(),
    end_date_raw=dbutils.widgets.get("end_date").strip(),
    lookback_days_raw=dbutils.widgets.get("lookback_days").strip(),
    run_id=dbutils.widgets.get("run_id").strip(),
    catalog=dbutils.widgets.get("catalog").strip() or "market_macro",
    display_fn=display,
)

result.as_dict()
